In [1]:
%load_ext autoreload
%autoreload 2

# dataIO: Positron focused (511 keV)
***

## Setup

### Imports

In [2]:
# Setting the root path for the imports
from pathlib import Path
import sys
working_dir=Path.cwd()
repo_root = (
    working_dir.parent if working_dir.name =='notebooks' else working_dir)
if repo_root not in sys.path: sys.path.insert(0,repo_root)

In [3]:

import numpy as np
import cosipy as cp
from cosipy.util import fetch_wasabi_file


### Paths to data files

In [ ]:
# Input directories
data_dir = repo_root / "data" / "inputs"

# Orientation data 
orientation_filename= "DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
data_ori_path = data_dir / orientation_filename


# Mock dataset: 14 weeks
weekly_files = {
    i: f"dc4_mock_dataset_week_{i}_unbinned_data_filtered_with_SAAcut.fits.gz"
    for i in range(1, 15)
}

weekly_paths = {
    week: data_dir / filename
    for week, filename in weekly_files.items()
}

# Response data
responses = ["Response511.o4.e509_513.s20881894470591.m2555.filtered.nonsparse.binnedimaging.imagingresponse.h5",
"ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5" ]

data_response511_path = data_dir / responses[0]
data_responsecon_path = data_dir / responses[1]

# extended response data 
extended_responses = ["extended_source_response_511_merged.h5.gz", "extended_source_response_continuum_merged.h5.gz"]

data_ext_response511_path = data_dir/extended_responses[0]
data_ext_responsecon_path = data_dir / extended_responses[1]


# Output directory
data_dir_out = repo_root / "data" / "output"
plot_dir_out = repo_root / "plot_outputs"

### Download Data
Skip this part if data already exist in the repository

In [5]:
def download_if_missing(remote_path, data_dir):
    filename = Path(remote_path).name
    local_file = data_dir / filename
   
    if local_file.exists():
        print(f"skipping {filename}: already exists")
        return

    fetch_wasabi_file(remote_path, output = str(data_dir))


In [ ]:
# download the data we will need for this notebook.


# Mock dataset: 14 weeks
for week, filename in weekly_files.items():
    download_if_missing(f"COSI-SMEX/DC4/Data/Mock_Dataset/{filename}", data_dir)

# Orientation data
download_if_missing(f"COSI-SMEX/DC4/Data/Orientation/{orientation_filename}",  data_dir)

# Responses
for filename in responses:
    
    download_if_missing(f"COSI-SMEX/DC4/Data/Responses/{filename}", data_dir)

# Extended responses
for filename in extended_responses:
    
    download_if_missing(f"COSI-SMEX/DC3/Data/Responses/extended_source_response/{filename}", data_dir)


skipping dc4_mock_dataset_week_1_unbinned_data_filtered_with_SAAcut.fits.gz: already exists
skipping dc4_mock_dataset_week_2_unbinned_data_filtered_with_SAAcut.fits.gz: already exists
skipping dc4_mock_dataset_week_3_unbinned_data_filtered_with_SAAcut.fits.gz: already exists
skipping dc4_mock_dataset_week_4_unbinned_data_filtered_with_SAAcut.fits.gz: already exists
skipping dc4_mock_dataset_week_5_unbinned_data_filtered_with_SAAcut.fits.gz: already exists
skipping dc4_mock_dataset_week_6_unbinned_data_filtered_with_SAAcut.fits.gz: already exists
skipping dc4_mock_dataset_week_7_unbinned_data_filtered_with_SAAcut.fits.gz: already exists
skipping dc4_mock_dataset_week_8_unbinned_data_filtered_with_SAAcut.fits.gz: already exists
skipping dc4_mock_dataset_week_9_unbinned_data_filtered_with_SAAcut.fits.gz: already exists
skipping dc4_mock_dataset_week_10_unbinned_data_filtered_with_SAAcut.fits.gz: already exists
skipping dc4_mock_dataset_week_11_unbinned_data_filtered_with_SAAcut.fits.gz: a

### Load and inspect response511
We need this step only if .yaml file is not created yet. Otherwise we can skip